In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('drive/MyDrive/AI+X고급/식물을 부탁해/kjw')

In [3]:
import json
import cv2
import random
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

In [4]:
# 음...이건 그냥 코드 돌릴때 일정하게 나올라고 설정한 거임.
# 1. 파이썬 내장 random 시드 고정
random.seed(42)

# 2. 넘파이 시드 고정
np.random.seed(42)

# 3. TensorFlow 시드 고정
tf.random.set_seed(42)

# 4. 환경 변수 (특히 GPU 연산의 비결정성 제어)
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TF_DETERMINISTIC_OPS"] = "1"  # TF 연산을 결정적으로

# 식물종 명시(필수!!!!)

In [5]:
plant_name = "스투키"

In [6]:
# ???_dataset_detection
# (1) 드라이브 → Colab으로 복사
!cp "/content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/dataset(detection)/스투키_dataset_detection.zip" "/content/"

# (2) Colab에서 압축 해제
!unzip -q "/content/스투키_dataset_detection.zip" -d "/content/스투키_dataset_detection"

# (3) 압축 파일 삭제 (용량 절약)
!rm "/content/스투키_dataset_detection.zip"

## 1. 객체 탐지(YOLO11n) 모델링

In [7]:
# YOLOv11 설치
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 38.8 MB/s eta 0:00:00


In [8]:
# -*- coding: utf-8 -*-
from ultralytics import YOLO
import os

data_yaml  = f"/content/{plant_name}_dataset_detection/data.yaml"
project    = f"yolo_log/{plant_name}_yolo/detect"

AUG = dict(
    hsv_h=0.015, hsv_s=0.60, hsv_v=0.60,
    degrees=5.0, translate=0.08, scale=0.30, shear=2.0,
    mosaic=1.0, mixup=0.15, copy_paste=0.15, close_mosaic=20,
    fliplr=0.5, flipud=0.0,
)

# --- Stage 1: 960 / 15 epochs ---
print("=== Stage 1: 960 / 15 epochs ===")
model = YOLO("yolo11n.pt")
stage1_name = "stage1_960"

results_stage1 = model.train(
    data=data_yaml,
    epochs=15,
    imgsz=960,
    batch=8,          # ★ 정수로 지정 (OOM 시 4 또는 2로 낮추기)
    workers=1,
    cache=True,
    device=0,
    project=project,
    name=stage1_name,
    exist_ok=True,
    **AUG
)

# --- Stage 2: 1280 / +10 epochs (total 25) ---
print("=== Stage 2: 1280 / +10 epochs (total 25) ===")
best_stage1 = os.path.join(project, stage1_name, "weights", "best.pt")
assert os.path.exists(best_stage1), f"Stage1 best.pt not found: {best_stage1}"

model1280 = YOLO(best_stage1)
stage2_name = "stage2_1280_ft"

results_stage2 = model1280.train(
    data=data_yaml,
    epochs=10,
    imgsz=1280,
    batch=4,          # ★ 1280은 메모리↑ → 더 낮게 (OOM 시 2 또는 1)
    workers=1,
    cache=True,
    device=0,
    project=project,
    name=stage2_name,
    exist_ok=True,
    **AUG
)

print("\n✅ Done.")
print(f"Stage1(best): {best_stage1}")
print(f"Stage2 runs : {os.path.join(project, stage2_name)}")
print("→ 최종 배포 후보 가중치: stage2_1280_ft/weights/best.pt (또는 last.pt)")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
=== Stage 1: 960 / 15 epochs ===
Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/스투키_dataset_detection/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.6, hsv_v=0.6,

In [9]:
# -----------------------------
# 최종 모델 검증(Validation)
#  - stage2(1280) best.pt 기준
# -----------------------------

best_stage2 = os.path.join(project, stage2_name, "weights", "best.pt")
assert os.path.exists(best_stage2), f"Stage2 best.pt not found: {best_stage2}"

val_model = YOLO(best_stage2)

metrics = val_model.val(
    data=data_yaml,          # 동일 data.yaml
    imgsz=1280,              # 학습 단계2와 동일 해상도
    batch=8,                 # ★ 정수로 지정 (OOM나면 4 또는 2로)
    device=0,
    project=project,
    name="val_stage2_1280",  # 결과 저장 폴더
    exist_ok=True,
    plots=True,              # PR/Confusion 등 그래프 저장
    save_json=True           # COCO 포맷 메트릭 JSON 저장
)
print(metrics)


Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2062.5±1687.1 MB/s, size: 1685.7 KB)
val: Scanning /content/스투키_dataset_detection/val/labels.cache... 1800 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1800/1800 3.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 225/225 7.7it/s 29.1s
                   all       1800       3600      0.996      0.997      0.995      0.968
Speed: 2.1ms preprocess, 8.0ms inference, 0.0ms loss, 0.8ms postprocess per image
Saving /content/drive/.shortcut-targets-by-id/1Rygpy_fXkK90skpjkGvVAK6_gxx8rqAN/식물을 부탁해/kjw/yolo_log/스투키_yolo/detect/val_stage2_1280/predictions.json...
Results saved to /content/drive/.shortcut-targets-by-id/1Rygpy_fXkK90skpjkGvVAK6_gxx8rqAN/식물을 부탁해/kjw/yolo_log/스투키_yolo/detect/

In [ ]:
# -----------------------------
# Test 세트 예측 (Prediction)
#  - stage2(1280) best.pt 기준
#  - 작은 잎까지 최대한 많이 탐지하도록 리콜 중심 파라미터
# -----------------------------

best_stage2 = os.path.join(project, stage2_name, "weights", "best.pt")
assert os.path.exists(best_stage2), f"Stage2 best.pt not found: {best_stage2}"

test_model = YOLO(best_stage2)

pred = test_model.predict(
    source=f"/content/{plant_name}_dataset_detection/test/images",  # 테스트 이미지 폴더
    data=data_yaml,
    save=True,                  # 탐지 결과 이미지 저장
    imgsz=1280,                 # 학습(Stage2)와 동일 해상도
    conf=0.18,                  # ↓ 낮춰서 작은 잎 후보 더 수집(0.18~0.25 범위 조정 가능)
    iou=0.70,                   # ↑ 겹친 잎 박스 유지 (NMS 덜 지우게)
    max_det=300,                # 이미지당 박스 상한 늘리기
    project=project,
    name="predict_stage2_1280", # 결과 폴더명
    exist_ok=True,
    save_txt=True,              # YOLO 포맷 라벨(txt) 저장
    save_conf=True,             # 각 박스 confidence 저장
    verbose=True
)

print("\n✅ Prediction 완료")
print(f"결과 폴더: {os.path.join(project, 'predict_stage2_1280')}")



WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1800 /content/스투키_dataset_detection/test/images/N50-A-1-01-B-1-H-230825-000004.jpg: 1280x736 2 leafs, 90.1ms
image 2/1800 /content/스투키_dataset_detection/test/images/N50-A-1-01-B-1-H-230826-000218.jpg: 1280x736 2 leafs, 14.7ms
image 3/1800 /content/스투키_dataset_detection/test/images/N50-A-1-01-B-1-H-230826-000236.jpg: 1280x736 2 leafs, 14.7ms
image 4/1800 /content/스투키_dataset_detection/test/images/N50-A-1-01-B-1-H-230826-000274.jpg: 1280x736 2 le

In [ ]:
# -----------------------------
# Test 세트 평가 (Evaluation)
#  - stage2(1280) best.pt 기준
# -----------------------------

# Stage2 best.pt 경로 지정
best_stage2 = os.path.join(project, stage2_name, "weights", "best.pt")
assert os.path.exists(best_stage2), f"Stage2 best.pt not found: {best_stage2}"

# 모델 로드
yolo_best_model = YOLO(best_stage2)

# 테스트 세트 평가
metrics = yolo_best_model.val(
    data=data_yaml,             # 동일한 data.yaml 사용
    split="test",               # test 세트 평가 명시
    imgsz=1280,                 # 학습과 동일한 해상도
    batch=8,                    # Colab GPU 여유에 따라 4~8로 조정
    conf=0.25,                  # 평가 기준 confidence threshold
    iou=0.5,                    # mAP50 기준 (COCO mAP50-95는 자동 계산)
    device=0,
    project=project,
    name="test_result_stage2_1280",  # 결과 폴더명
    exist_ok=True,
    plots=True,                 # PR curve, confusion matrix 등 저장
    save_json=True              # COCO 포맷 결과 JSON 저장
)

print("\n✅ Test 세트 평가 완료")
print(metrics)
print(f"결과 폴더: {os.path.join(project, 'test_result_stage2_1280')}")


Epoch: 현재 학습 중인 Epoch 번호

GPU_mem: 현재 GPU 메모리 사용량

box_loss / cls_loss / dfl_loss: 학습 손실(Loss) 값

 box_loss: 바운딩 박스 회귀 손실

 cls_loss: 클래스 분류 손실 (이번 경우엔 leaf=0 하나라서 점점 작아짐)

 dfl_loss: 분포 초점 손실 (YOLO의 anchor-free bounding box 품질)


Instances: 배치에 들어간 객체(바운딩박스) 개수

Size: 입력 이미지 크기 (ex. 640×640)

P (Precision)
→ 탐지한 것 중에서 맞은 비율 (정밀도)

R (Recall)
→ 실제 객체 중에서 탐지한 비율 (재현율)

mAP50 (mean Average Precision at IoU=0.5)
→ 가장 대표적인 정확도 지표.
값이 1.0(100%)이면 거의 완벽한 탐지 성능.

mAP50-95
→ IoU 0.5부터 0.95까지 평균낸 AP. 더 엄격한 지표.
보통 이 값이 모델의 실질적인 성능 평가 기준.